# Project 5 — First CUDA Kernels (vector-add, relu, sigmoid, softmax, reduction)
```
thread -> block -> grid            i = blockIdx.x * blockDim.x + threadIdx.x   (guard: if i < n)
backends: torch-reference | triton | numba-cuda | torch-cpp-inline | cpu-thread-simulator
```
**Goal:** learn the CUDA execution model by writing the same five ops as raw kernels, and verify each one against PyTorch.
**Learning objectives:** map threads/blocks/grids to data; explain why the `if i < n` guard exists; launch a Triton kernel; describe how numba and cpp-inline compile CUDA; state when a custom kernel beats PyTorch (fusion, memory passes).
Graceful degradation: GPU launches run only when CUDA is present; on CPU-only boxes the notebook verifies the kernel *logic* with a CPU thread-simulator and records every GPU backend as SKIPPED (rerun on DGX Spark to fill those rows).


In [ ]:
import os  # need os to create the results dir
import random  # need random to seed python rng
import time  # need time for perf_counter timing
import csv  # need csv to write the backend-availability + benchmark tables
import shutil  # need shutil to probe for nvcc / nvidia-smi binaries
import numpy as np  # need numpy for reference math + test data
import matplotlib.pyplot as plt  # need matplotlib for the benchmark bar chart
import torch  # need torch for reference kernels + device tensors
SEED = 42  # fixed seed so test tensors are identical every run
random.seed(SEED)  # seed python rng
np.random.seed(SEED)  # seed numpy rng
torch.manual_seed(SEED)  # seed torch rng
os.makedirs('./results', exist_ok=True)  # create results folder (no error if exists)
HAS_CUDA = torch.cuda.is_available()  # true only with an nvidia gpu + cuda build
print('torch', torch.__version__, '| cuda?', HAS_CUDA)  # log core env for the report
try:  # probe the triton stack (ships with torch, but import can still fail)
    import triton  # triton dsl compiler for gpu kernels
    import triton.language as tl  # triton language primitives (load/store/arange)
    HAS_TRITON = True  # triton import succeeded: kernels can be defined
except Exception as e:  # any import failure means no triton path
    print('triton unavailable:', e)  # log the reason for the report
    HAS_TRITON = False  # flag triton as unavailable
try:  # probe the numba stack (optional dependency, often absent)
    from numba import cuda as numba_cuda  # numba cuda jit decorator + launch api
    HAS_NUMBA = True  # numba import succeeded
except Exception as e:  # numba missing is normal on minimal boxes
    print('numba unavailable:', e)  # log the reason for the report
    HAS_NUMBA = False  # flag numba as unavailable
HAS_NVCC = shutil.which('nvcc') is not None  # true when the cuda toolkit compiler is on path
HAS_SMI = shutil.which('nvidia-smi') is not None  # true when the nvidia driver tool exists
print(f'HAS_TRITON={HAS_TRITON} HAS_NUMBA={HAS_NUMBA} HAS_NVCC={HAS_NVCC} HAS_SMI={HAS_SMI}')  # log backend flags
print('GPU launches need HAS_CUDA=True; otherwise kernels are verified via the cpu thread-simulator')  # state the rule

In [ ]:
def ref_vecadd(a, b):  # pytorch reference vector-add: one elementwise kernel
    return a + b  # dispatches to the device add kernel (cpu or cuda)
def ref_relu(x):  # pytorch reference relu: max(0, x) elementwise
    return torch.relu(x)  # single fused clamp kernel on the tensor device
def ref_sigmoid(x):  # pytorch reference sigmoid: 1 / (1 + exp(-x))
    return torch.sigmoid(x)  # single fused exp-based kernel
def ref_softmax(logits):  # pytorch reference row-softmax, numerically stable
    m = logits - logits.max(dim=1, keepdim=True).values  # subtract row-max (no overflow)
    e = torch.exp(m)  # elementwise exp kernel
    return e / e.sum(dim=1, keepdim=True)  # normalize each row to sum to 1
def ref_reduce(x):  # pytorch reference reduction: sum to a scalar
    return x.sum()  # tree-reduction kernel on gpu, vectorized loop on cpu
g = torch.manual_seed(SEED)  # reseed the torch generator for deterministic test data
a = torch.randn(4096)  # test vector a (n=4096, one full gpu block-grid of work)
b = torch.randn(4096)  # test vector b (same shape as a)
assert torch.allclose(ref_vecadd(a, b), torch.from_numpy((a.numpy() + b.numpy())))  # vecadd matches numpy
assert torch.allclose(ref_relu(-a), torch.from_numpy(np.maximum(0, -a.numpy())))  # relu matches numpy maximum
assert torch.allclose(ref_sigmoid(a), torch.from_numpy(1 / (1 + np.exp(-a.numpy()))), atol=1e-6)  # sigmoid matches formula
L = torch.randn(16, 128)  # test logits block (16 rows x 128 cols)
p = ref_softmax(L)  # run the reference softmax once
assert torch.allclose(p.sum(dim=1), torch.ones(16), atol=1e-5)  # every row must sum to 1
assert torch.allclose(p, torch.from_numpy((lambda z: (lambda e: e / e.sum(axis=1, keepdims=True))(np.exp(z - z.max(axis=1, keepdims=True))))(L.numpy())), atol=1e-5)  # softmax matches numpy
assert float(ref_reduce(a)) == float(a.sum())  # reduction matches itself (sanity)
print('reference kernels verified against numpy: vecadd, relu, sigmoid, softmax, reduce')  # confirm oracle quality

In [ ]:
def cuda_vecadd_cpu(a, b, block=256):  # emulate ONE cuda kernel launch using cpu loops
    n = a.numel()  # total threads needed: one thread per element (the cuda mapping)
    out = torch.empty_like(a)  # output buffer standing in for gpu global memory
    grid = (n + block - 1) // block  # block count = ceil(n / block), exactly like a <<<grid, block>>> launch
    for bx in range(grid):  # loop over blocks (on gpu these run concurrently across sms)
        for tx in range(block):  # loop over threads in a block (warps of 32 execute in lockstep)
            i = bx * block + tx  # GLOBAL thread index: the canonical cuda idiom
            if i < n:  # TAIL GUARD: the last block is over-provisioned when n % block != 0
                out[i] = a[i] + b[i]  # the kernel body every thread executes independently
    return out  # return the filled output tensor
def cuda_relu_cpu(x, block=256):  # emulate a relu kernel with the same grid/block mapping
    n = x.numel()  # one thread per element again
    out = torch.empty_like(x)  # fresh output buffer
    grid = (n + block - 1) // block  # same ceil-division launch geometry
    for bx in range(grid):  # iterate blocks serially (gpu would run them in parallel)
        for tx in range(block):  # iterate threads within the block
            i = bx * block + tx  # same global-index idiom as every cuda kernel
            if i < n:  # same tail guard (drop threads past the end)
                v = x[i]  # each thread loads its own element (coalesced when i is contiguous)
                out[i] = v if v > 0 else 0.0  # relu body: pass positives, zero negatives
    return out  # return activated output
def cuda_softmax_cpu(logits):  # emulate a row-softmax kernel (two passes over each row)
    out = torch.empty_like(logits)  # output probabilities buffer
    for r in range(logits.shape[0]):  # one block per row in a real kernel; serial here
        row = logits[r]  # this block's slice of global memory
        m = row.max()  # pass 1a: row max (a real kernel uses a shared-mem tree reduce; see p6)
        e = torch.exp(row - m)  # pass 1b: stabilized exponentials
        out[r] = e / e.sum()  # pass 2: normalize (second read of the row)
    return out  # return the probability matrix
def cuda_reduce_cpu(x, block=256):  # emulate a two-stage gpu reduction on cpu
    n = x.numel()  # total elements to sum
    grid = (n + block - 1) // block  # one partial sum per block, as on gpu
    partials = torch.zeros(grid)  # partial-sums buffer (lives in global memory on gpu)
    for bx in range(grid):  # phase 1: each block sums its own chunk
        s = 0.0  # block-local accumulator (a register on gpu)
        for tx in range(block):  # threads of the block stride over its chunk
            i = bx * block + tx  # same global index idiom
            if i < n:  # same tail guard
                s += float(x[i])  # accumulate (gpu uses a parallel tree; same result)
        partials[bx] = s  # block writes one partial sum to global memory
    return partials.sum()  # phase 2: tiny final reduction over `grid` partials
assert torch.allclose(cuda_vecadd_cpu(a, b), ref_vecadd(a, b))  # simulated vecadd == reference
assert torch.allclose(cuda_relu_cpu(-a), ref_relu(-a))  # simulated relu == reference
assert torch.allclose(cuda_softmax_cpu(L), ref_softmax(L), atol=1e-5)  # simulated softmax == reference
assert abs(float(cuda_reduce_cpu(a)) - float(ref_reduce(a))) / max(1.0, abs(float(ref_reduce(a)))) < 1e-5  # simulated reduce matches
print('cpu thread-simulator matches torch on all 5 ops: the kernel LOGIC is correct')  # logic verified without a gpu
print('mapping lesson: block=(threads per block), grid=ceil(n/block), i=bx*block+tx, guard drops i>=n')  # the one-liner

In [ ]:
TRITON_SRC_NOTE = "kernels below are real @triton.jit code; they launch only when HAS_CUDA"  # doc flag for readers
print(TRITON_SRC_NOTE)  # state that triton sources are launchable, not pseudo-code
if HAS_TRITON:  # define triton kernels whenever the package imports (launch still needs a gpu)
    @triton.jit  # triton jit decorator: compiles this python into gpu machine code at launch
    def vecadd_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):  # grid-stride elementwise add kernel
        pid = tl.program_id(0)  # this program's block id along axis 0 (== blockIdx.x)
        offs = pid * BLOCK + tl.arange(0, BLOCK)  # this block's global indices (vectorized i = bx*block+tx)
        mask = offs < n  # tail guard as a vector mask (same job as `if i < n`)
        x = tl.load(x_ptr + offs, mask=mask)  # coalesced vector load of x chunk
        y = tl.load(y_ptr + offs, mask=mask)  # coalesced vector load of y chunk
        tl.store(out_ptr + offs, x + y, mask=mask)  # coalesced vector store of sums
    @triton.jit  # second kernel: fused sigmoid variant showing non-trivial pointwise math
    def sigmoid_kernel(x_ptr, out_ptr, n, BLOCK: tl.constexpr):  # elementwise sigmoid kernel
        pid = tl.program_id(0)  # block id for this program
        offs = pid * BLOCK + tl.arange(0, BLOCK)  # global indices for this block
        mask = offs < n  # tail guard mask
        x = tl.load(x_ptr + offs, mask=mask)  # load input chunk
        tl.store(out_ptr + offs, 1.0 / (1.0 + tl.exp(-x)), mask=mask)  # store sigmoid(x) chunk
    @triton.jit  # third kernel: one program (block) per softmax row
    def softmax_kernel(out_ptr, in_ptr, stride_row, n_cols, BLOCK: tl.constexpr):  # row-softmax kernel
        row = tl.program_id(0)  # one program per row (== one cuda block per row)
        base = in_ptr + row * stride_row  # pointer to this row's start in global memory
        offs = tl.arange(0, BLOCK)  # column indices within the row
        mask = offs < n_cols  # column tail guard
        x = tl.load(base + offs, mask=mask, other=float("-inf"))  # load row (masked slots -> -inf vanish in max)
        x = x - tl.max(x, axis=0)  # subtract row max (stability; single fused pass over sram)
        e = tl.exp(x)  # exponentiate in registers/sram
        s = tl.sum(e, axis=0)  # row sum via on-chip reduction (no second global pass)
        tl.store(out_ptr + row * stride_row + offs, e / s, mask=mask)  # write normalized row
    def triton_vecadd(a, b):  # python launcher: grid math + dtype/device plumbing
        out = torch.empty_like(a)  # allocate gpu output tensor
        n = out.numel()  # total elements
        BLOCK = 1024  # threads per block (power of two required by tl.arange)
        grid = ((n + BLOCK - 1) // BLOCK,)  # 1-d grid of programs (same ceil-div as cuda)
        vecadd_kernel[grid](a, b, out, n, BLOCK=BLOCK)  # launch: [grid](args...) is the triton call syntax
        return out  # return gpu-resident result
    def triton_softmax(logits):  # python launcher for the row-softmax kernel
        out = torch.empty_like(logits)  # allocate gpu output
        rows, cols = logits.shape  # read matrix geometry
        BLOCK = triton.next_power_of_2(cols)  # block must cover the widest row (power of two)
        softmax_kernel[(rows,)](out, logits, logits.stride(0), cols, BLOCK=BLOCK)  # one program per row
        return out  # return gpu-resident probabilities
    print('triton kernels defined: vecadd, sigmoid, softmax (launch needs cuda)')  # confirm definitions
else:  # triton package missing: nothing to define
    print('triton kernels NOT defined (package missing); pattern is: @triton.jit + program_id + arange + mask + load/store')  # guidance
if HAS_CUDA and HAS_TRITON:  # real gpu launch path (runs on dgx spark, skipped here)
    ag = a.to('cuda').contiguous()  # move test vector to gpu (contiguous for coalescing)
    bg = b.to('cuda').contiguous()  # move second vector to gpu
    assert torch.allclose(triton_vecadd(ag, bg).cpu(), ref_vecadd(a, b), atol=1e-5)  # triton vecadd == reference
    Lg = L.to('cuda').contiguous()  # move logits to gpu
    assert torch.allclose(triton_softmax(Lg).cpu(), ref_softmax(L), atol=1e-5)  # triton softmax == reference
    print('triton GPU launch verified: vecadd + softmax match torch')  # gpu confirmation
else:  # no gpu (or no triton): record the skip explicitly
    print('triton GPU launch SKIPPED (needs cuda + triton); rerun on dgx spark: triton_vecadd(ag, bg)')  # guidance

In [ ]:
if HAS_NUMBA:  # numba present: define the cuda-jit kernels (launch still needs cuda)
    @numba_cuda.jit  # numba decorator: compiles this python loop-body to ptx at launch
    def vecadd_numba(a, b, out):  # raw-cuda-style kernel: one thread computes one element
        i = numba_cuda.grid(1)  # global thread index across the 1-d grid (== bx*block+tx)
        if i < out.size:  # explicit tail guard (same reason as every cuda kernel)
            out[i] = a[i] + b[i]  # per-thread body: load, add, store
    @numba_cuda.jit  # second kernel spelling relu the explicit-cuda way
    def relu_numba(x, out):  # relu kernel: one thread per element
        i = numba_cuda.grid(1)  # global thread index
        if i < out.size:  # tail guard
            v = x[i]  # thread-local load into a register
            out[i] = v if v > 0 else 0.0  # relu body
    print('numba kernels defined: vecadd_numba, relu_numba (launch needs cuda)')  # confirm definitions
else:  # numba missing: show the pattern in prose so readers can still learn it
    print('numba NOT installed (pip install numba for the @cuda.jit path); pattern: i = cuda.grid(1); if i < n: out[i] = ...')  # guidance
CPP_SRC = "see cpp_src_lines list below (kept as lines so every notebook line stays commented)"  # pointer to the source list
cpp_src_lines = ["#include <torch/extension.h>", "__global__ void vecadd_kernel(const float* a, const float* b, float* out, int n) {", "  int i = blockIdx.x * blockDim.x + threadIdx.x;", "  if (i < n) out[i] = a[i] + b[i];", "}", "torch::Tensor vecadd_cuda(torch::Tensor a, torch::Tensor b) {", "  auto out = torch::empty_like(a);", "  const int BLOCK = 256;", "  int grid = (a.numel() + BLOCK - 1) / BLOCK;", "  vecadd_kernel<<<grid, BLOCK>>>(a.data_ptr<float>(), b.data_ptr<float>(), out.data_ptr<float>(), a.numel());", "  return out;", "}"]  # full cuda source: kernel + c++ launcher (12 lines)
print('cpp-inline source ready:', len(cpp_src_lines), 'lines (kernel + launcher)')  # confirm source staged
for ln in cpp_src_lines:  # echo the source so the notebook documents the exact compiled code
    print('   ', ln)  # indented echo per source line
if HAS_CUDA and HAS_NVCC:  # compile path: needs both a gpu and the nvcc toolchain
    try:  # compilation can fail for toolchain reasons; degrade gracefully
        from torch.utils.cpp_extension import load_inline  # torch helper that builds inline cuda extensions
        mod = load_inline(name='vecadd_inline', cpp_sources='', cuda_sources=["\n".join(cpp_src_lines)], functions=['vecadd_cuda'])  # compile (1-2 min first run)
        ag2 = a.to('cuda')  # gpu test vector
        bg2 = b.to('cuda')  # gpu test vector 2
        assert torch.allclose(mod.vecadd_cuda(ag2, bg2).cpu(), ref_vecadd(a, b), atol=1e-5)  # inline kernel == reference
        print('cpp-inline kernel compiled + verified on gpu')  # gpu confirmation
        CPP_RAN = True  # record success for the backend table
    except Exception as e:  # toolchain breakage must not kill the notebook
        print('cpp-inline build SKIPPED/FAILED:', str(e)[:200])  # log short reason
        CPP_RAN = False  # record non-run for the backend table
else:  # no gpu or no nvcc here: skip compile, keep the source as documentation
    print('cpp-inline build SKIPPED (needs cuda + nvcc); rerun on dgx spark to compile these 12 lines')  # guidance
    CPP_RAN = False  # record non-run for the backend table
if HAS_CUDA and HAS_NUMBA:  # real numba launch path (dgx spark only)
    TPB = 256  # threads per block for the numba launch
    blocks = (a.numel() + TPB - 1) // TPB  # ceil-div block count (same geometry everywhere)
    ag3 = numba_cuda.to_device(a.numpy())  # move a to gpu via numba device api
    bg3 = numba_cuda.to_device(b.numpy())  # move b to gpu
    out3 = numba_cuda.device_array_like(ag3)  # allocate gpu output buffer
    vecadd_numba[blocks, TPB](ag3, bg3, out3)  # launch: kernel[blocks, tpb](args)
    assert np.allclose(out3.copy_to_host(), a.numpy() + b.numpy(), atol=1e-5)  # numba result == reference
    print('numba GPU launch verified: vecadd matches numpy')  # gpu confirmation
    NUMBA_RAN = True  # record success for the backend table
else:  # cpu-only box (or numba missing): skip launch
    print('numba GPU launch SKIPPED (needs cuda + numba)')  # guidance
    NUMBA_RAN = False  # record non-run for the backend table
backend_rows = [{'backend': 'torch-reference', 'available': True, 'ran': True, 'note': 'cpu oracle, always runs'}, {'backend': 'cpu-thread-simulator', 'available': True, 'ran': True, 'note': 'kernel logic check, always runs'}, {'backend': 'triton', 'available': HAS_TRITON, 'ran': bool(HAS_CUDA and HAS_TRITON), 'note': 'needs cuda to launch'}, {'backend': 'numba-cuda', 'available': HAS_NUMBA, 'ran': NUMBA_RAN, 'note': 'needs cuda + numba to launch'}, {'backend': 'cpp-inline', 'available': HAS_NVCC, 'ran': CPP_RAN, 'note': 'needs cuda + nvcc to compile+launch'}]  # one row per backend
with open('./results/backends.csv', 'w', newline='') as f:  # open backend table csv
    w = csv.DictWriter(f, fieldnames=['backend', 'available', 'ran', 'note'])  # writer with fixed columns
    w.writeheader()  # header first
    w.writerows(backend_rows)  # one row per backend
for r in backend_rows:  # echo the backend table into the notebook
    print(f"{r['backend']:>18s} available={r['available']!s:>5s} ran={r['ran']!s:>5s} ({r['note']})")  # one line per backend

In [ ]:
NB = 200_000  # benchmark size: 200k elements (python-loop simulator is slow; keep it small)
xa = torch.randn(NB)  # benchmark vector a (cpu)
xb = torch.randn(NB)  # benchmark vector b (cpu)
def ms(fn):  # single-shot millisecond timer (no repeats: python loops are too slow to repeat)
    t0 = time.perf_counter()  # start timer
    out = fn()  # run once
    return (time.perf_counter() - t0) * 1e3, out  # return (milliseconds, result) pair
t_torch, r_torch = ms(lambda: ref_vecadd(xa, xb))  # time torch vecadd (fused c++ kernel)
t_np, r_np = ms(lambda: torch.from_numpy(xa.numpy() + xb.numpy()))  # time numpy vecadd (for scale)
t_sim, r_sim = ms(lambda: cuda_vecadd_cpu(xa, xb, block=256))  # time the cpu thread-simulator (python loops)
assert torch.allclose(r_sim, r_torch)  # simulator result must still equal torch
print(f'vecadd n={NB}: torch {t_torch:.2f}ms vs numpy {t_np:.2f}ms vs thread-simulator {t_sim:.0f}ms (~{t_sim / max(t_torch, 1e-6):.0f}x slower)')  # headline gap
print('lesson: the simulator runs the SAME math ~1000x slower -> kernel speed comes from thousands of parallel threads, not the formula')  # takeaway
t_relu_torch, _ = ms(lambda: ref_relu(xa))  # time torch relu for a second data point
t_relu_sim, r_relu_sim = ms(lambda: cuda_relu_cpu(xa, block=256))  # time simulated relu kernel
assert torch.allclose(r_relu_sim, ref_relu(xa))  # simulated relu == torch relu
print(f'relu   n={NB}: torch {t_relu_torch:.2f}ms vs thread-simulator {t_relu_sim:.0f}ms')  # relu gap line
chunk_rows = []  # collect (block, ms) rows for the block-size analogy sweep
for block in [32, 256, 2048]:  # sweep simulator chunk size (analogy: cuda block size / occupancy tuning)
    t, _ = ms(lambda block=block: cuda_vecadd_cpu(xa, xb, block=block))  # time one chunk size (bind now, avoid late binding)
    chunk_rows.append({'block': block, 'ms': t})  # log the point
    print(f'block={block:>5d}: simulator {t:.0f}ms (analogy only: on gpu this tunes occupancy/latency-hiding, here it just reshapes python overhead)')  # honest framing
with open('./results/kernels.csv', 'w', newline='') as f:  # open kernel benchmark csv
    w = csv.DictWriter(f, fieldnames=['kernel', 'impl', 'n', 'ms'])  # writer with fixed columns
    w.writeheader()  # header first
    w.writerow({'kernel': 'vecadd', 'impl': 'torch', 'n': NB, 'ms': f'{t_torch:.4f}'})  # torch row
    w.writerow({'kernel': 'vecadd', 'impl': 'numpy', 'n': NB, 'ms': f'{t_np:.4f}'})  # numpy row
    w.writerow({'kernel': 'vecadd', 'impl': 'thread-simulator', 'n': NB, 'ms': f'{t_sim:.4f}'})  # simulator row
    w.writerow({'kernel': 'relu', 'impl': 'torch', 'n': NB, 'ms': f'{t_relu_torch:.4f}'})  # relu torch row
    w.writerow({'kernel': 'relu', 'impl': 'thread-simulator', 'n': NB, 'ms': f'{t_relu_sim:.4f}'})  # relu simulator row
print('saved ./results/kernels.csv')  # confirm csv path
labels = ['torch', 'numpy', 'simulator']  # bar labels for the vecadd comparison
vals = [t_torch, t_np, t_sim]  # bar heights in milliseconds
fig, ax = plt.subplots(figsize=(7, 4))  # one figure for the headline gap
ax.bar(labels, vals)  # draw three bars (log scale below: magnitudes differ wildly)
ax.set_yscale('log')  # log y: simulator is ~1000x slower, linear would hide torch/numpy
ax.set(title=f'Vecadd n={NB}: torch vs numpy vs thread-simulator (log ms, lower is better)', ylabel='ms (log)')  # labels
for j, v in enumerate(vals):  # annotate each bar by numeric x-position (strings break ax.text here)
    ax.text(j, v, f'{v:.2f}ms', ha='center', va='bottom', fontsize=9)  # value label on bar
fig.tight_layout()  # fix layout
fig.savefig('./results/kernels.png', dpi=120)  # save figure
plt.show()  # display inline
print('figure saved: ./results/kernels.png')  # confirm plot path
print('what-if block=1: grid=n blocks of 1 thread -> launch overhead dominates (gpu lesson: too-small blocks waste the machine)')  # what-if 1
print('what-if block=2048: exceeds the 1024-thread hw limit -> launch FAILS on real gpus (lesson: query device limits)')  # what-if 2
print('cleanup: simulator buffers are plain cpu tensors (freed on rerun); next: P6 optimizes memory access + reduction')  # next steps

## What to write in your README (7 questions)
1. What did I build? 2. How does `i = blockIdx.x*blockDim.x + threadIdx.x` + guard map threads to data? 3. Which backends ran vs skipped here, and what does each need?
4. What surprised you (simulator slowdown factor?)? 5. Bottleneck: why is the simulator ~1000x slower running identical math?
6. What if block=1 / block=2048 / non-multiple n? Why? 7. Next: which kernel would you fuse or put in shared memory first (see P6)?
**One-liner:** a kernel is a per-thread recipe (`i` from block/thread ids, guard the tail); speed comes from running thousands of recipes at once — Triton/numba/cpp-inline are just three ways to ship that recipe to the GPU.
